In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [2]:
data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [3]:
#pandas loads in the data as a nice dataframe
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [54]:
# we actually want to work with a numpy array to be able manipulate and do fancy algebra
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev   = data[0:1000].T  #first 1000 examples and transpose it so the first row is the target for ease of access
Y_dev      = data_dev[0]
X_dev      = (data_dev[1:n] / 255.0).astype(np.float32) #numpy defaults to float64, which is slower and uses more memory

data_train = data[1000:m].T
Y_train    = data_train[0]
X_train    = (data_train[1:n] / 255.0).astype(np.float32)

In [55]:
Y_train #all targets

array([0, 0, 2, ..., 6, 4, 7])

In [56]:
X_train[:, 0].shape #first column should be all pixels of the first number

(784,)

In [57]:
def init_params():
    W1 = np.random.randn(10,784) * 0.01 #get random nums between -0.5 and 0.5 and multiply by 0.01 to make weights tiny (better for ReLU)
    b1 = np.zeros((10,1)) #initalizing all biases to 0 is standard

    W2 = np.random.randn(10, 10) * 0.01
    b2 = np.zeros((10,1))
    return W1, b1, W2, b2

In [14]:
def ReLU(Z):
    return np.maximum(0, Z) #element wise, so it will go through each element in Z, and anything <= 0 will be turned into 0

In [47]:
def softmax(Z):
    Z = Z - Z.max(axis=0)
    return np.exp(Z) / np.sum(np.exp(Z), axis=0) #exp will do e^z for every z in Z. sum will preserve columns and collapse rows to get 1 sum

In [44]:
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1 #take the dot product of the weight matrix and input vector then add bias vector
    A1 = ReLU(Z1) #activate it!
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

In [18]:
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1)) #y.size is # of examples, take max of 0-9 (which is 9) and add 1 to get 10 output classes
    one_hot_Y[np.arange(Y.size), Y] = 1 #for each row, go to the column specified by the label in y and set it to 1\
    one_hot_Y = one_hot_Y.T #right now each row is an example, but we want it the other way around
    return one_hot_Y

In [20]:
def ReLU_prime(Z):
    return Z > 0

In [40]:
def back_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    
    dZ1 = W2.T.dot(dZ2) * ReLU_prime(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

In [22]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate):
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    return W1, b1, W2, b2

In [23]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

In [49]:
def gradient_descent(X, Y, epochs, learning_rate):
    W1, b1, W2, b2 = init_params()
    for i in range(epochs):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, learning_rate)
        if i % 50 == 0:
            print("Iteration: ", i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))
    return W1, b1, W2, b2

In [58]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 500, 0.1)

Iteration:  0
[2 4 4 ... 4 4 2] [0 0 2 ... 6 4 7]
Accuracy:  0.08695121951219512
Iteration:  50
[0 0 0 ... 3 0 0] [0 0 2 ... 6 4 7]
Accuracy:  0.2059512195121951
Iteration:  100
[0 0 0 ... 3 6 7] [0 0 2 ... 6 4 7]
Accuracy:  0.5672439024390243
Iteration:  150
[0 0 2 ... 1 4 7] [0 0 2 ... 6 4 7]
Accuracy:  0.7674390243902439
Iteration:  200
[0 0 2 ... 1 4 7] [0 0 2 ... 6 4 7]
Accuracy:  0.8198780487804878
Iteration:  250
[0 0 2 ... 1 4 7] [0 0 2 ... 6 4 7]
Accuracy:  0.8445365853658536
Iteration:  300
[0 0 2 ... 1 4 7] [0 0 2 ... 6 4 7]
Accuracy:  0.8610975609756097
Iteration:  350
[0 0 2 ... 1 4 7] [0 0 2 ... 6 4 7]
Accuracy:  0.872219512195122
Iteration:  400
[0 0 2 ... 1 4 7] [0 0 2 ... 6 4 7]
Accuracy:  0.8809512195121951
Iteration:  450
[0 0 2 ... 6 4 7] [0 0 2 ... 6 4 7]
Accuracy:  0.8865609756097561
